In [1]:
import os
import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression

In [2]:
DATA_PATH = "../data/processed/cleaned_data.csv"

df = pd.read_csv(DATA_PATH)

print("Dataset loaded successfully!")
print("Shape:", df.shape)

Dataset loaded successfully!
Shape: (2830743, 79)


In [3]:
PREPROCESSING_DIR = "../models/preprocessing"

scaler = joblib.load(
    os.path.join(PREPROCESSING_DIR, "scaler.joblib")
)

label_encoder = joblib.load(
    os.path.join(PREPROCESSING_DIR, "label_encoder.joblib")
)

print("Scaler loaded:", type(scaler).__name__)
print("Label encoder loaded:", type(label_encoder).__name__)

Scaler loaded: StandardScaler
Label encoder loaded: LabelEncoder


In [4]:
FEATURES = [
    "Destination Port",
    "Flow Duration",
    "Total Fwd Packets",
    "Total Length of Fwd Packets",
    "Fwd Packet Length Max",
    "Fwd Packet Length Min",
    "Fwd Packet Length Mean",
    "Bwd Packet Length Max",
    "Bwd Packet Length Min",
    "Flow Bytes/s",
    "Flow Packets/s",
    "Flow IAT Mean",
    "Flow IAT Std",
    "Flow IAT Max",
    "Flow IAT Min",
    "Fwd IAT Mean",
    "Fwd IAT Std",
    "Fwd IAT Min",
    "Bwd IAT Total",
    "Bwd IAT Mean",
    "Bwd IAT Std",
    "Bwd IAT Max",
    "Bwd IAT Min",
    "Fwd PSH Flags",
    "Fwd URG Flags",
    "Bwd Packets/s",
    "Min Packet Length",
    "Max Packet Length",
    "Packet Length Mean",
    "Packet Length Variance",
    "FIN Flag Count",
    "RST Flag Count",
    "PSH Flag Count",
    "ACK Flag Count",
    "URG Flag Count",
    "ECE Flag Count",
    "Down/Up Ratio",
    "Init_Win_bytes_forward",
    "Init_Win_bytes_backward",
    "min_seg_size_forward",
    "Active Mean",
    "Active Std",
    "Active Max",
    "Active Min",
    "Idle Std"
]

print("Number of features:", len(FEATURES))

Number of features: 45


In [5]:
X = df[FEATURES]
y = label_encoder.transform(df["Label"])

print("Feature matrix:", X.shape)
print("Target:", y.shape)

Feature matrix: (2830743, 45)
Target: (2830743,)


In [6]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=42,
    stratify=y
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    random_state=42,
    stratify=y_temp
)

print("Training:", X_train.shape)
print("Validation:", X_val.shape)
print("Test:", X_test.shape)

Training: (1981520, 45)
Validation: (424611, 45)
Test: (424612, 45)


In [7]:
X_train_scaled = scaler.transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

print("Scaled training:", X_train_scaled.shape)
print("Scaled validation:", X_val_scaled.shape)
print("Scaled test:", X_test_scaled.shape)

Scaled training: (1981520, 45)
Scaled validation: (424611, 45)
Scaled test: (424612, 45)


In [8]:
logistic_model = LogisticRegression(
    max_iter=1000,
    solver="lbfgs",
    n_jobs=-1
)

In [9]:
logistic_model = LogisticRegression(
    max_iter=1000,
    solver="lbfgs",
    n_jobs=-1
)

print("Starting Logistic Regression training...")

logistic_model.fit(
    X_train_scaled,
    y_train
)

print("Logistic Regression training completed!")

Starting Logistic Regression training...


f:\NIDS-ML-NEW\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1435: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)


Logistic Regression training completed!


In [10]:
from sklearn.metrics import accuracy_score, classification_report

print("Evaluating Logistic Regression...")

y_val_pred = logistic_model.predict(X_val_scaled)

accuracy = accuracy_score(y_val, y_val_pred)

print("Validation Accuracy:", accuracy)

print("\nClassification Report:")
print(
    classification_report(
        y_val,
        y_val_pred,
        target_names=label_encoder.classes_,
        zero_division=0
    )
)

Evaluating Logistic Regression...
Validation Accuracy: 0.9744754610690727

Classification Report:
                            precision    recall  f1-score   support

                    BENIGN       0.99      0.98      0.98    340964
                       Bot       0.75      0.01      0.02       295
                      DDoS       1.00      0.95      0.97     19204
             DoS GoldenEye       0.95      0.82      0.88      1544
                  DoS Hulk       0.94      1.00      0.97     34661
          DoS Slowhttptest       0.82      0.80      0.81       825
             DoS slowloris       0.88      0.55      0.68       870
               FTP-Patator       0.96      1.00      0.98      1190
                Heartbleed       1.00      1.00      1.00         2
              Infiltration       0.33      0.17      0.22         6
                  PortScan       0.82      1.00      0.90     23839
               SSH-Patator       0.97      0.47      0.64       884
  Web Attack - Br

In [11]:
MODEL_DIR = "../models"

os.makedirs(MODEL_DIR, exist_ok=True)

logistic_path = os.path.join(
    MODEL_DIR,
    "logistic_regression.joblib"
)

joblib.dump(
    logistic_model,
    logistic_path
)

print("Logistic Regression model saved successfully!")
print("Path:", logistic_path)

Logistic Regression model saved successfully!
Path: ../models\logistic_regression.joblib


In [12]:
print(
    "Model exists:",
    os.path.exists(logistic_path)
)

Model exists: True


In [13]:
from sklearn.ensemble import RandomForestClassifier

print("Random Forest imported successfully!")

Random Forest imported successfully!


In [14]:
random_forest_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=None,
    min_samples_split=2,
    random_state=42,
    n_jobs=-1
)

print("Random Forest model created successfully!")

Random Forest model created successfully!


In [15]:
print("Starting Random Forest training...")

random_forest_model.fit(
    X_train,
    y_train
)

print("Random Forest training completed!")

Starting Random Forest training...
Random Forest training completed!


In [16]:
from sklearn.metrics import accuracy_score, classification_report

print("Evaluating Random Forest...")

y_val_pred_rf = random_forest_model.predict(X_val)

accuracy_rf = accuracy_score(y_val, y_val_pred_rf)

print("Validation Accuracy:", accuracy_rf)

print("\nClassification Report:")
print(
    classification_report(
        y_val,
        y_val_pred_rf,
        target_names=label_encoder.classes_,
        zero_division=0
    )
)

Evaluating Random Forest...
Validation Accuracy: 0.9986481744467289

Classification Report:
                            precision    recall  f1-score   support

                    BENIGN       1.00      1.00      1.00    340964
                       Bot       0.88      0.78      0.83       295
                      DDoS       1.00      1.00      1.00     19204
             DoS GoldenEye       1.00      0.99      1.00      1544
                  DoS Hulk       1.00      1.00      1.00     34661
          DoS Slowhttptest       1.00      0.99      0.99       825
             DoS slowloris       0.99      1.00      0.99       870
               FTP-Patator       1.00      1.00      1.00      1190
                Heartbleed       1.00      1.00      1.00         2
              Infiltration       1.00      0.67      0.80         6
                  PortScan       0.99      1.00      1.00     23839
               SSH-Patator       1.00      1.00      1.00       884
  Web Attack - Brute Fo

In [17]:
MODEL_DIR = "../models/supervised"

os.makedirs(MODEL_DIR, exist_ok=True)

random_forest_path = os.path.join(
    MODEL_DIR,
    "random_forest.joblib"
)

joblib.dump(
    random_forest_model,
    random_forest_path
)

print("Random Forest model saved successfully!")
print("Path:", random_forest_path)

Random Forest model saved successfully!
Path: ../models/supervised\random_forest.joblib


In [18]:
print(
    "Model exists:",
    os.path.exists(random_forest_path)
)

Model exists: True
